# Week 16, Day 3: Memoisation

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w016d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
calls = 0
def fib(n):
    global calls
    calls += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)

fib(20)
print(calls)

In [ ]:
memo = {}

def fib(n):
    if n in memo:
        return memo[n]
    answer = n if n < 2 else fib(n - 1) + fib(n - 2)
    memo[n] = answer
    return answer

print(fib(20), fib(90))
print(len(memo), "answers remembered")

*Interactive figure on the website: With memoisation, a call that has been made before returns at once: its subtree is never explored again. Compare fib(6) here with yesterday's tree..*

In [ ]:
def count_paths(n):
    memo = {}
    def go(k):
        if k == 0:
            return 1
        if k < 0:
            return 0
        if k not in memo:
            memo[k] = go(k - 1) + go(k - 2)
        return memo[k]
    return go(n)

print(count_paths(80))

> **Key idea.** Memoisation: before working anything out, check a dictionary of answers already found; after working it out, save it there. Each different argument is then worked out exactly once.

In [ ]:
from functools import cache

@cache
def ways(amount, coins):
    if amount == 0:
        return 1
    if amount < 0 or not coins:
        return 0
    return ways(amount - coins[0], coins) + ways(amount, coins[1:])

print(ways(200, (1, 2, 5, 10, 20, 50, 100, 200)))

> **Common mistake.** `@cache` on a function that takes a list fails with `TypeError: unhashable type: 'list'`. Pass `tuple(items)` instead, or memoise on something hashable that describes the problem, such as a position in the list.

In [ ]:
def fib_table(n):
    table = [0, 1] + [0] * max(0, n - 1)
    for k in range(2, n + 1):
        table[k] = table[k - 1] + table[k - 2]
    return table[n]

print(fib_table(90), fib_table(0), fib_table(1))

In [ ]:
def ways_table(amount, coins):
    table = [1] + [0] * amount
    for coin in coins:
        for a in range(coin, amount + 1):
            table[a] += table[a - coin]
    return table[amount]

print(ways_table(500, [1, 2, 5, 10, 20, 50, 100, 200]))

In [ ]:
import time
from functools import cache

def slow(n):
    return n if n < 2 else slow(n - 1) + slow(n - 2)

@cache
def fast(n):
    return n if n < 2 else fast(n - 1) + fast(n - 2)

for f in (slow, fast):
    start = time.perf_counter()
    f(25)
    print(f.__name__, f"{time.perf_counter() - start:.5f} s")

### Exercise: Memoise by hand

Add memoisation to `trib(n)` (the tribonacci numbers: each is the sum of the three before, starting 0, 0, 1) with a dictionary, **without** using `functools`. `trib(200)` must be quick.

In [ ]:
def trib(n):
    if n < 2:
        return 0
    if n == 2:
        return 1
    return trib(n - 1) + trib(n - 2) + trib(n - 3)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"fast and right\"\n    assert [trib(n) for n in range(10)] == [0, 0, 1, 1, 2, 4, 7, 13, 24, 44]\n    assert trib(200) % 1000 == 832\n    assert \"functools\" not in source and \"cache\" not in source.replace(\"memo\", \"\")")

### Exercise: Big grids

Use `@cache` to make `routes(rows, cols)` (the number of right-and-down routes through a grid, from yesterday) fast enough for a 17 by 17 grid.

In [ ]:
from functools import cache

def routes(rows, cols):
    if rows == 1 or cols == 1:
        return 1
    return routes(rows - 1, cols) + routes(rows, cols - 1)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"big grids\"\n    assert routes(3, 3) == 6 and routes(17, 17) == 601080390 and routes(1, 50) == 1\n    assert \"@cache\" in source or \"lru_cache\" in source")

### Exercise: Positions, not slices

Rewrite `ways(amount, coins)` so that it memoises on a position: write a helper `go(a, i)` that counts the ways to make `a` using coins from position `i` onwards, decorated with `@cache`. `ways` takes a list and must work for `ways(200, [1, 2, 5, 10, 20, 50, 100, 200])`, the number of ways to make two pounds from British coins.

In [ ]:
from functools import cache

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert ways(10, [1, 2, 5]) == 10\n    assert ways(0, [3]) == 1 and ways(7, []) == 0\n    assert ways(200, [1, 2, 5, 10, 20, 50, 100, 200]) == 73682")

### Exercise: Bottom up

Write `climb_table(n, steps)` that counts the ways to climb `n` stairs using any of the step sizes in `steps` (order matters), bottom-up with a table and no recursion. `climb_table(4, [1, 2])` is 5.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert climb_table(4, [1, 2]) == 5\n    assert [climb_table(n, [1, 2, 3]) for n in range(8)] == [1, 1, 2, 4, 7, 13, 24, 44]\n    assert climb_table(10, [3, 5]) == 1 and climb_table(0, [2]) == 1\n    assert climb_table(1000, [1, 2]) % 10 ** 6 == 403501\n    assert \"climb_table(\" not in source.split(\"def climb_table\", 1)[1]")

### Exercise: Longest common subsequence

A **subsequence** of a string keeps some of its characters in order, perhaps with gaps: `"ace"` is a subsequence of `"abcde"`. The longest common subsequence of two strings is used in `diff` tools, spell checkers and DNA comparison. Write `lcs_length(a, b)` with memoised recursion on positions `i` and `j`: if `a[i] == b[j]`, the answer is 1 plus the answer for `i + 1, j + 1`; otherwise it's the larger of skipping a character from `a` or from `b`.

In [ ]:
from functools import cache

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert lcs_length(\"abcde\", \"ace\") == 3\n    assert lcs_length(\"abc\", \"def\") == 0\n    assert lcs_length(\"\", \"abc\") == 0\n    assert lcs_length(\"AGGTAB\", \"GXTXAYB\") == 4\n    assert lcs_length(\"python\" * 30, \"typhon\" * 30) > 100")

### Bug hunt: A cache that lies

`next_ticket()` should give a new ticket number every time it's called: 1, 2, 3. Someone added `@cache` to "speed it up", and now it gives 1 every time. Fix it.

In [ ]:
from functools import cache

counter = [0]

@cache
def next_ticket():
    counter[0] += 1
    return counter[0]

print(next_ticket(), next_ticket(), next_ticket())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"new numbers\"\n    expect_output(\"1 2 3\")")

### Bug hunt: Unhashable

This crashes with `TypeError: unhashable type: 'list'`. Fix it so the call works, without removing the cache.

In [ ]:
from functools import cache

@cache
def total(values):
    if not values:
        return 0
    return values[0] + total(values[1:])

print(total([3, 4, 5]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"12\"\n    expect_output(\"12\")\n    assert \"@cache\" in source")

### Puzzle: Memoised climbing

Arrange the lines for a memoised `paths(n)`, counting climbs of 1 or 2 stairs at a time. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # with memoisation, how many different values of n does fib(30) work out? (0 to 30 inclusive)
b = ___   # ways_table(5, [1, 2])
c = ___   # True or False: a function that returns random.randint(1, 6) is safe to cache
d = ___   # fib_table(10)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 31 and b == 3 and c is False and d == 55")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: edit distance

The **edit distance** (Levenshtein distance) between two words is the fewest single-character edits (insert, delete or replace a character) that turn one into the other. "kitten" to "sitting" takes 3: replace k with s, replace e with i, insert g. Spell checkers use it to suggest the nearest real word. Write `edit_distance(a, b)` with memoised recursion on positions, then `closest(word, dictionary)` that returns the dictionary word with the smallest edit distance from `word` (the alphabetically first if tied).

The rule, for positions `i` in `a` and `j` in `b`: if either word is used up, the answer is the number of characters left in the other. If `a[i] == b[j]`, the answer is the answer for `i + 1, j + 1`. Otherwise it's 1 plus the smallest of: delete (`i + 1, j`), insert (`i, j + 1`), replace (`i + 1, j + 1`).

In [ ]:
from functools import cache

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"distances\"\n    assert edit_distance(\"kitten\", \"sitting\") == 3\n    assert edit_distance(\"\", \"abc\") == 3 and edit_distance(\"abc\", \"abc\") == 0\n    assert edit_distance(\"flaw\", \"lawn\") == 2\n    assert edit_distance(\"intention\", \"execution\") == 5\ndef test_closest():\n    \"closest\"\n    words = [\"python\", \"pythons\", \"typhoon\", \"pylon\", \"piston\"]\n    assert closest(\"pythn\", words) == \"python\"\n    assert closest(\"pyton\", words) == \"pylon\"\n    assert closest(\"pistol\", words) == \"piston\"\n    assert closest(\"pylo\", words) == \"pylon\"")

## Warm-up

Over 20,000 calls to work out 21 different values (`fib(0)` to `fib(20)`). Every value after the first few is worked out hundreds or thousands of times. If each one were worked out **once**, `fib(20)` would need about 20 calls of real work.

## Remembering answers

**Memoisation** (from "memo", a note to yourself) means: keep a dictionary from arguments to answers. Before doing any work, look the arguments up; if they're there, return the saved answer. Otherwise work it out, save it, and return it.

`fib(90)` is instant now, where the plain version would take longer than a lifetime. Step through both and compare the trees:

(If you trace twice, the second trace starts with the memo already filled from the first, so it returns at once. Edit the code, or press Trace after changing the call, to start fresh.)

The memo usually belongs **inside** a function, so separate uses don't share it by accident. A helper does it neatly:

## functools.cache

Memoisation is so useful that Python has it built in. Put `@cache` (from the `functools` module) on the line above the function, and Python keeps the memo dictionary for you:

The `@cache` line is a **decorator**: it wraps the function in another that does the looking up and saving. You'll learn to write decorators in Phase 4; for now, use this one.

Two things to notice in that example. The coins are a **tuple**, not a list: the arguments become a dictionary key, so they must be hashable. And `ways(200, ...)` (the ways to make two pounds) answers instantly; without the cache it makes over 22 million calls.

`lru_cache(maxsize=1000)` is the same, but it forgets the least recently used answers once it holds `maxsize` of them, which matters when there are millions of possible arguments and memory is limited.

## What can be memoised

Memoisation saves the answer the **first** time and returns it ever after. That's only correct when the answer depends on nothing but the arguments:

- **Pure functions**: same arguments, same answer, no side effects. `fib`, `ways`, `routes`: perfect.
- **Not**: functions that read the time, a file, user input, a random number, or a global variable that changes. A cached `roll_dice()` would return the same roll for ever.
- **Not**: functions whose job is an effect, like printing or appending to a list. The effect would only happen once.
- **Arguments must be hashable**: numbers, strings, tuples. Not lists or dictionaries (convert to a tuple first).

A common, efficient pattern is to memoise on **positions** rather than slices: instead of `ways(amount, coins[1:])`, use `ways(amount, i)` meaning "using coins from position `i` onwards". Positions are numbers, so they're hashable, and no slices are copied.

## Bottom up

Memoisation works **top-down**: start from the big problem, and only the smaller problems that are actually needed get solved. The other way is **bottom-up**: solve the smallest problems first, in a table, then use them to build the next ones, with a plain loop and no recursion.

Each entry is filled from entries already filled. This style is called **dynamic programming**, a name Richard Bellman chose in the 1950s; memoisation and bottom-up tables are its two forms.

Here's counting change bottom-up. `table[a]` is the number of ways to make amount `a`; adding coin types one at a time keeps the "order doesn't matter" rule:

| | Top-down (memoised recursion) | Bottom-up (table) |
|---|---|---|
| Written as | the recursive definition, plus a memo | a loop filling a table |
| Solves | only the subproblems that are needed | every subproblem up to n |
| Depth | can hit the recursion limit for large n | no recursion at all |
| Usually easier to | write first | make fast and small |

A good habit: write the plain recursion, add `@cache`, and only write a table if you need to avoid the recursion limit or save memory.

## Measuring

From exponential to linear: `slow(25)` makes about 240,000 calls and `fast(25)` makes 26. The bigger n gets, the more dramatic the gap.

## Practice

## Quiz

## Daily challenge

## Before you go

Take two tree-recursive functions from yesterday and add `@cache` to each; then write one of them bottom-up with a table. Time all three versions on a big input. Tomorrow: permutations and subsets, where the number of answers really is enormous, and no cache can help.